# R3 — Universal Circuits (Threshold Sweep, Rust)

Mirrors `3_universals.ipynb` for Rust data from R2.
Applies 9 (ε × consistency) thresholds, computes universal construct/object circuits.

In [1]:
# Cell 1 – Configuration
import subprocess, sys, os

pkgs = ["h5py", "numpy", "pandas", "tqdm"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

EPSILONS = [0.001, 0.1, 0.5]

LANG = "R"
MODEL = "DS"
PREFIX = f"{LANG}_{MODEL}_"
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
CONSISTENCIES = [0.2, 0.5, 0.8]
N_LAYERS = MODEL_CONFIGS[MODEL]["n_layers"]
OBJECT_ACTIVATIONS = f"{PREFIX}2_object_activations.h5"
CHECKER_ACTIVATIONS = f"{PREFIX}2_checker_activations.h5"

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/New-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/New-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Grid: {len(EPSILONS)} eps x {len(CONSISTENCIES)} cons = {len(EPSILONS)*len(CONSISTENCIES)} combos")

Environment: Local
Grid: 3 eps x 3 cons = 9 combos


In [2]:
# Cell 2 – Imports & load raw activations
import numpy as np
import h5py
from tqdm.auto import tqdm
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("R3")

from module2.io_utils import load_activations_hdf5
from module2.marginalization import UniversalModuleComputer
from module2.metrics import compute_jaccard_matrix

obj_data = load_activations_hdf5(os.path.join(DATA_DIR, OBJECT_ACTIVATIONS))
obj_activations = obj_data["pair_activations"]
print(f"Object activations: {len(obj_activations)} pairs")

chk_data = load_activations_hdf5(os.path.join(DATA_DIR, CHECKER_ACTIVATIONS))
chk_activations = chk_data["pair_activations"]
print(f"Checker activations: {len(chk_activations)} pairs")

INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:module2.io_utils:Loaded activations: 690 pairs


Object activations: 690 pairs


INFO:module2.io_utils:Loaded activations: 63 pairs


Checker activations: 63 pairs


In [3]:
# Cell 3 – Threshold function

def apply_thresholds(pair_activations, epsilon, consistency):
    """Apply epsilon + consistency thresholds to raw activations."""
    pair_masks = {}
    for key, raw in pair_activations.items():
        n_prompts = raw["n_prompts"]
        masks = {}
        for lid, layer_data in raw["layers"].items():
            mean_act = layer_data["activation_sum"] / n_prompts
            firing_rate = layer_data["firing_count"].astype(np.float32) / n_prompts
            mask = (mean_act > epsilon) & (firing_rate >= consistency)
            masks[lid] = mask
        pair_masks[key] = masks
    return pair_masks

print("Threshold function defined.")

Threshold function defined.


In [4]:
# Cell 4 – Object sweep (with universal marginalization)
#
# Key difference from Python: save universal masks with "rust__" prefix
# so they match R1B checker keys ("rust__For", "rust__Vec", etc.)

computer = UniversalModuleComputer()
constructs = sorted(set(c for c, _ in obj_activations))
objects = sorted(set(o for _, o in obj_activations))
print(f"Constructs: {len(constructs)}, Objects: {len(objects)}")

for eps in EPSILONS:
    for cons in CONSISTENCIES:
        print(f"\n--- Object: eps={eps}, cons={cons} ---")
        pair_masks = apply_thresholds(obj_activations, eps, cons)

        # compute_all returns {"ast": {construct: {layer: mask}}, "builtin": {object: {layer: mask}}}
        # We pass constructs as "ast_nodes" and objects as "builtin_objs" — the function is generic
        universal_masks = computer.compute_all(pair_masks, constructs, objects)

        # Metrics
        rep_layer = 14
        metrics = {}
        if universal_masks.get("ast"):
            # Wrap in the format compute_jaccard_matrix expects: {name: {layer: mask}}
            metrics["jaccard_construct_matrix"] = compute_jaccard_matrix(universal_masks["ast"], rep_layer)
        if universal_masks.get("builtin"):
            metrics["jaccard_object_matrix"] = compute_jaccard_matrix(universal_masks["builtin"], rep_layer)

        # Save manually with "rust__" prefix for both constructs and objects
        out_name = f"{PREFIX}3_object_masks_eps{eps}_cons{cons}.h5"
        out_path = os.path.join(DATA_DIR, out_name)

        with h5py.File(out_path, "w") as f:
            # Pair masks
            pg = f.create_group("pair_masks")
            for (c, o), layers in pair_masks.items():
                for lid, mask in layers.items():
                    pg.create_dataset(f"layer_{lid}/{c}__{o}",
                                      data=mask.astype(np.bool_), compression="gzip")

            # Universal masks with rust__ prefix
            ug = f.create_group("universal_masks")
            # Construct universals: "rust__For", "rust__If", etc.
            for name, layers in universal_masks.get("ast", {}).items():
                for lid, mask in layers.items():
                    ug.create_dataset(f"layer_{lid}/rust__{name}",
                                      data=mask.astype(np.bool_), compression="gzip")
            # Object universals: "rust__i32", "rust__Vec", etc.
            # Skip if name collides with a construct universal (e.g., Fn)
            for name, layers in universal_masks.get("builtin", {}).items():
                for lid, mask in layers.items():
                    ds_path = f"layer_{lid}/rust__{name}"
                    if ds_path not in ug:
                        ug.create_dataset(ds_path,
                                          data=mask.astype(np.bool_), compression="gzip")

            # Metrics
            mg = f.create_group("metrics")
            for k, v in metrics.items():
                mg.create_dataset(k, data=v)

            # Metadata
            md = f.create_group("metadata")
            md.attrs["epsilon"] = eps
            md.attrs["consistency_thresh"] = cons
            md.attrs["n_pairs"] = len(pair_masks)
            md.attrs["n_layers"] = N_LAYERS
            md.create_dataset("constructs", data=np.array(constructs, dtype=h5py.string_dtype()))
            md.create_dataset("objects", data=np.array(objects, dtype=h5py.string_dtype()))

        n_construct_universals = len(universal_masks.get("ast", {}))
        n_object_universals = len(universal_masks.get("builtin", {}))
        print(f"  Saved: {out_name} ({n_construct_universals} construct + {n_object_universals} object universals)")

Constructs: 34, Objects: 42

--- Object: eps=0.001, cons=0.2 ---


INFO:module2.marginalization:Universal AST As: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Async: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Await: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Break: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Closure: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Const: intersection across 20/42 builtins
INFO:module2.marginalization:Universal AST Continue: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Crate: intersection across 1/42 builtins
INFO:module2.marginalization:Universal AST Dyn: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Enum: intersection across 29/42 builtins
INFO:module2.marginalization:Universal AST Fn: intersection across 30/42 builtins
INFO:module2.marginalization:Universal AST For: intersection across 24

  Saved: R_DS_3_object_masks_eps0.001_cons0.2.h5 (34 construct + 42 object universals)

--- Object: eps=0.001, cons=0.5 ---


INFO:module2.marginalization:Universal AST As: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Async: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Await: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Break: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Closure: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Const: intersection across 20/42 builtins
INFO:module2.marginalization:Universal AST Continue: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Crate: intersection across 1/42 builtins
INFO:module2.marginalization:Universal AST Dyn: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Enum: intersection across 29/42 builtins
INFO:module2.marginalization:Universal AST Fn: intersection across 30/42 builtins
INFO:module2.marginalization:Universal AST For: intersection across 24

  Saved: R_DS_3_object_masks_eps0.001_cons0.5.h5 (34 construct + 42 object universals)

--- Object: eps=0.001, cons=0.8 ---


...

  Saved: R_DS_3_object_masks_eps0.5_cons0.2.h5 (34 construct + 42 object universals)

--- Object: eps=0.5, cons=0.5 ---


INFO:module2.marginalization:Universal AST As: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Async: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Await: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Break: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Closure: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Const: intersection across 20/42 builtins
INFO:module2.marginalization:Universal AST Continue: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Crate: intersection across 1/42 builtins
INFO:module2.marginalization:Universal AST Dyn: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Enum: intersection across 29/42 builtins
INFO:module2.marginalization:Universal AST Fn: intersection across 30/42 builtins
INFO:module2.marginalization:Universal AST For: intersection across 24

  Saved: R_DS_3_object_masks_eps0.5_cons0.5.h5 (34 construct + 42 object universals)

--- Object: eps=0.5, cons=0.8 ---


INFO:module2.marginalization:Universal AST As: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Async: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Await: intersection across 14/42 builtins
INFO:module2.marginalization:Universal AST Break: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Closure: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Const: intersection across 20/42 builtins
INFO:module2.marginalization:Universal AST Continue: intersection across 15/42 builtins
INFO:module2.marginalization:Universal AST Crate: intersection across 1/42 builtins
INFO:module2.marginalization:Universal AST Dyn: intersection across 32/42 builtins
INFO:module2.marginalization:Universal AST Enum: intersection across 29/42 builtins
INFO:module2.marginalization:Universal AST Fn: intersection across 30/42 builtins
INFO:module2.marginalization:Universal AST For: intersection across 24

  Saved: R_DS_3_object_masks_eps0.5_cons0.8.h5 (34 construct + 42 object universals)


In [5]:
# Cell 5 – Checker sweep (no marginalization)

for eps in EPSILONS:
    for cons in CONSISTENCIES:
        print(f"\n--- Checker: eps={eps}, cons={cons} ---")
        checker_masks = apply_thresholds(chk_activations, eps, cons)

        out_name = f"{PREFIX}3_checker_masks_eps{eps}_cons{cons}.h5"
        out_path = os.path.join(DATA_DIR, out_name)

        with h5py.File(out_path, "w") as f:
            tcm = f.create_group("token_checker_masks")
            for (obj_type, obj_name), layers in checker_masks.items():
                obj_key = f"{obj_type}__{obj_name}"  # e.g., "rust__For"
                for lid, mask in layers.items():
                    tcm.create_dataset(
                        f"layer_{lid}/{obj_key}",
                        data=mask.astype(np.bool_),
                        compression="gzip"
                    )
            md = f.create_group("metadata")
            md.attrs["epsilon"] = eps
            md.attrs["consistency_threshold"] = cons

        print(f"  Saved: {out_name} ({len(checker_masks)} objects)")


--- Checker: eps=0.001, cons=0.2 ---
  Saved: R_DS_3_checker_masks_eps0.001_cons0.2.h5 (63 objects)

--- Checker: eps=0.001, cons=0.5 ---
  Saved: R_DS_3_checker_masks_eps0.001_cons0.5.h5 (63 objects)

--- Checker: eps=0.001, cons=0.8 ---
  Saved: R_DS_3_checker_masks_eps0.001_cons0.8.h5 (63 objects)

--- Checker: eps=0.1, cons=0.2 ---
  Saved: R_DS_3_checker_masks_eps0.1_cons0.2.h5 (63 objects)

--- Checker: eps=0.1, cons=0.5 ---
  Saved: R_DS_3_checker_masks_eps0.1_cons0.5.h5 (63 objects)

--- Checker: eps=0.1, cons=0.8 ---
  Saved: R_DS_3_checker_masks_eps0.1_cons0.8.h5 (63 objects)

--- Checker: eps=0.5, cons=0.2 ---
  Saved: R_DS_3_checker_masks_eps0.5_cons0.2.h5 (63 objects)

--- Checker: eps=0.5, cons=0.5 ---
  Saved: R_DS_3_checker_masks_eps0.5_cons0.5.h5 (63 objects)

--- Checker: eps=0.5, cons=0.8 ---
  Saved: R_DS_3_checker_masks_eps0.5_cons0.8.h5 (63 objects)


In [6]:
# Cell 6 – Summary
print(f"\nTotal: {2 * len(EPSILONS) * len(CONSISTENCIES)} HDF5 files in {DATA_DIR}")
print(f"  R3_object_masks: {len(EPSILONS) * len(CONSISTENCIES)} files")
print(f"  R3_checker_masks: {len(EPSILONS) * len(CONSISTENCIES)} files")
print("\nR3 complete.")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")


Total: 18 HDF5 files in /Users/piotrwilam/Data/New-Atlas
  R3_object_masks: 9 files
  R3_checker_masks: 9 files

R3 complete.
Not running on Colab -- skipping unassign.
